In [ ]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
print(pytesseract.get_tesseract_version())


### Dataset

Load dataset and visualise it : I choosed: - [FUNSD](https://huggingface.co/datasets/nielsr/funsd)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("nielsr/funsd")
print(dataset)

In [ ]:
import numpy as np
sample = dataset["train"][0]
sample.keys()

In [ ]:
print(type(sample["image"]))
print(np.array(sample["image"]).shape)
print(sample["words"][:5])
print(sample["bboxes"][:5])

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Convert PIL → NumPy
image = np.array(sample["image"])
if len(image.shape) == 2:  # grayscale
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

h, w = image.shape[:2]
bboxes = sample["bboxes"]
words = sample["words"]

for bbox, word in zip(bboxes, words):
    x0, y0, x1, y1 = bbox
    x0 = int(x0 * w / 1000)
    x1 = int(x1 * w / 1000)
    y0 = int(y0 * h / 1000)
    y1 = int(y1 * h / 1000)

    cv2.rectangle(image, (x0, y0), (x1, y1), (0, 0, 255), 1)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,10))
plt.imshow(image_rgb)
plt.axis("off")
plt.show()


In [ ]:
#Check label mapping
label_feature = dataset["train"].features["ner_tags"].feature
label_names = label_feature.names
print(label_names)


### Model

In [8]:
import json
import re
import numpy as np
from datasets import load_dataset
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from datetime import datetime

import numpy as np
import cv2
from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from datetime import datetime



Load LayoutLMv3 Processor

In [9]:
class FUNSDDocumentProcessor:
    """
    Document understanding pipeline using FUNSD dataset
    Uses pre-annotated text and bounding boxes (no OCR needed)
    """

    def __init__(self):
        print("="*70)
        print("FUNSD DOCUMENT UNDERSTANDING PIPELINE")
        print("="*70)

        # Load FUNSD dataset
        self.dataset = self.load_funsd()

        # FUNSD label mapping
        self.label_map = {
            0: "O",           # Outside
            1: "B-QUESTION",  # Beginning of question
            2: "I-QUESTION",  # Inside question
            3: "B-ANSWER",    # Beginning of answer
            4: "I-ANSWER",    # Inside answer
            5: "B-HEADER",    # Beginning of header
            6: "I-HEADER"     # Inside header
        }

        print(f"✓ Loaded FUNSD dataset")
        print(f"  Training samples: {len(self.dataset['train'])}")
        print(f"  Test samples: {len(self.dataset['test'])}")

    def load_funsd(self):
        """Load FUNSD dataset from HuggingFace"""
        try:
            dataset = load_dataset("nielsr/funsd")
            return dataset
        except Exception as e:
            print(f"Error loading FUNSD: {e}")
            # Fallback to local data or create synthetic
            return self.create_synthetic_data()


    def visualize_document(self, doc, show_words=False):
        """
        Visualize FUNSD document with correct colors + entity annotations
        """

        # Image conversion
        image = np.array(doc["image"])

        # Grayscale → BGR
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

        # BGR → RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        img_h, img_w = image.shape[:2]

        words = doc["words"]
        bboxes = doc["bboxes"]
        ner_tags = doc["ner_tags"]

        fig, axes = plt.subplots(1, 2, figsize=(16, 9))

        # -----------------------
        # Original image
        # -----------------------
        axes[0].imshow(image)
        axes[0].set_title(f"Original Document (ID: {doc['id']})")
        axes[0].axis("off")

        # -----------------------
        # Annotated image
        # -----------------------
        axes[1].imshow(image)

        colors = {
            "HEADER": "blue",
            "QUESTION": "red",
            "ANSWER": "green",
            "O": "gray"
        }

        for word, bbox, tag_idx in zip(words, bboxes, ner_tags):

            label_name = self.label_map[tag_idx]
            entity = label_name.split("-")[-1] if label_name != "O" else "O"
            color = colors.get(entity, "gray")

            # Scale bbox from 0–1000 → pixels
            x0 = int(bbox[0] * img_w / 1000)
            y0 = int(bbox[1] * img_h / 1000)
            x1 = int(bbox[2] * img_w / 1000)
            y1 = int(bbox[3] * img_h / 1000)

            rect = patches.Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                linewidth=2,
                edgecolor=color,
                facecolor="none"
            )
            axes[1].add_patch(rect)

            if show_words and entity != "O":
                axes[1].text(
                    x0,
                    y0 - 4,
                    word,
                    fontsize=8,
                    color=color,
                    bbox=dict(facecolor="white", alpha=0.6, edgecolor="none")
                )

        axes[1].set_title("Layout + Semantic Labels")
        axes[1].axis("off")

        # Legend
        legend = [
            patches.Patch(color="blue", label="HEADER"),
            patches.Patch(color="red", label="QUESTION"),
            patches.Patch(color="green", label="ANSWER"),
            patches.Patch(color="gray", label="O"),
        ]
        axes[1].legend(handles=legend, loc="lower right")

        plt.tight_layout()
        plt.show()

        return fig


    def process_document(self, doc):
        """
        Main processing pipeline for a FUNSD document

        Args:
            doc: FUNSD document with 'words', 'bboxes', 'ner_tags', 'image'

        Returns:
            Structured information extracted from the document
        """

        print(f"\n{'='*60}")
        print(f"PROCESSING DOCUMENT: {doc['id']}")
        print(f"{'='*60}")

        print("\n[1/4] Analyzing document structure...")
        layout_data = self.analyze_layout(doc)

        print("[2/4] Extracting entities...")
        entities = self.extract_entities(doc)

        print("[3/4] Extracting key-value pairs...")
        key_value_pairs = self.extract_key_value_pairs(doc)

        print("[4/4] Building structured output...")
        structured_data = self.build_structured_output(doc, layout_data, entities, key_value_pairs)

        return structured_data

    def analyze_layout(self, doc):
        """Analyze document layout based on bounding boxes"""

        words = doc['words']
        bboxes = doc['bboxes']
        image = doc['image']

        height, width = image.size[1], image.size[0]

        # Initialize layout sections
        layout = {
            'header': {'bboxes': [], 'words': [], 'indices': []},
            'left_column': {'bboxes': [], 'words': [], 'indices': []},
            'right_column': {'bboxes': [], 'words': [], 'indices': []},
            'body': {'bboxes': [], 'words': [], 'indices': []},
            'footer': {'bboxes': [], 'words': [], 'indices': []}
        }

        # Analyze each word's position
        for i, (word, bbox) in enumerate(zip(words, bboxes)):
            if not word.strip():
                continue

            # Calculate center position
            x_center = (bbox[0] + bbox[2]) / 2
            y_center = (bbox[1] + bbox[3]) / 2

            # Normalize positions
            x_norm = x_center / 1000
            y_norm = y_center / 1000

            # Categorize based on position
            if y_norm < 0.2:  # Top 20% = header
                layout['header']['indices'].append(i)
                layout['header']['bboxes'].append(bbox)
                layout['header']['words'].append(word)
            elif y_norm > 0.8:  # Bottom 20% = footer
                layout['footer']['indices'].append(i)
                layout['footer']['bboxes'].append(bbox)
                layout['footer']['words'].append(word)
            elif x_norm < 0.4:  # Left 40% = left column
                layout['left_column']['indices'].append(i)
                layout['left_column']['bboxes'].append(bbox)
                layout['left_column']['words'].append(word)
            elif x_norm > 0.6:  # Right 40% = right column
                layout['right_column']['indices'].append(i)
                layout['right_column']['bboxes'].append(bbox)
                layout['right_column']['words'].append(word)
            else:  # Middle = body
                layout['body']['indices'].append(i)
                layout['body']['bboxes'].append(bbox)
                layout['body']['words'].append(word)

        # Print layout summary
        print(f"   Header: {len(layout['header']['words'])} words")
        print(f"   Left column: {len(layout['left_column']['words'])} words")
        print(f"   Right column: {len(layout['right_column']['words'])} words")
        print(f"   Body: {len(layout['body']['words'])} words")
        print(f"   Footer: {len(layout['footer']['words'])} words")

        return layout

    def extract_entities(self, doc):
        """Extract entities from FUNSD-style BIO labels Keeps text + bounding boxes"""

        words = doc["words"]
        bboxes = doc["bboxes"]
        ner_tags = doc["ner_tags"]

        entities = {
            "HEADER": [],
            "QUESTION": [],
            "ANSWER": [],
            "O": []
        }

        current_type = None
        current_words = []
        current_boxes = []

        for word, box, tag_idx in zip(words, bboxes, ner_tags):
            label = self.label_map[tag_idx]

            if label.startswith("B-"):
                # save previous entity
                if current_type is not None:
                    entities[current_type].append({
                        "text": " ".join(current_words),
                        "bboxes": current_boxes
                    })

                current_type = label.split("-")[1]
                current_words = [word]
                current_boxes = [box]

            elif label.startswith("I-"):
                entity_type = label.split("-")[1]

                if current_type == entity_type:
                    current_words.append(word)
                    current_boxes.append(box)
                else:
                    # Handle broken BIO (treat as B-)
                    current_type = entity_type
                    current_words = [word]
                    current_boxes = [box]

            else:  # "O"
                if current_type is not None:
                    entities[current_type].append({
                        "text": " ".join(current_words),
                        "bboxes": current_boxes
                    })
                    current_type = None
                    current_words = []
                    current_boxes = []

                entities["O"].append(word)

        # flush last entity
        if current_type is not None:
            entities[current_type].append({
                "text": " ".join(current_words),
                "bboxes": current_boxes
            })

        return entities


    def extract_key_value_pairs(self, doc):
        """Extract question-answer pairs from the document"""

        words = doc['words']
        bboxes = doc['bboxes']
        ner_tags = doc['ner_tags']

        key_value_pairs = []
        current_question = []
        current_answer = []
        current_question_bbox = None

        # Group words by their NER tags
        for i, (word, bbox, tag_idx) in enumerate(zip(words, bboxes, ner_tags)):
            label = self.label_map.get(tag_idx, "O")

            if label.startswith('B-QUESTION') or label == 'I-QUESTION':
                if current_answer and current_question:
                    # Save previous pair
                    pair = self.create_key_value_pair(
                        current_question, current_answer, current_question_bbox, bbox
                    )
                    key_value_pairs.append(pair)
                    current_answer = []

                current_question.append(word)
                if label.startswith('B-QUESTION'):
                    current_question_bbox = bbox

            elif label.startswith('B-ANSWER') or label == 'I-ANSWER':
                current_answer.append(word)

            elif label == "O" and current_question and current_answer:
                # End of a pair
                pair = self.create_key_value_pair(
                    current_question, current_answer, current_question_bbox, bbox
                )
                key_value_pairs.append(pair)
                current_question = []
                current_answer = []
                current_question_bbox = None

        # Handle last pair
        if current_question and current_answer:
            pair = self.create_key_value_pair(
                current_question, current_answer, current_question_bbox, bboxes[-1]
            )
            key_value_pairs.append(pair)

        return key_value_pairs

    def create_key_value_pair(self, question_words, answer_words, question_bbox, answer_bbox):
        """Create a structured key-value pair"""

        question = ' '.join(question_words)
        answer = ' '.join(answer_words)

        # Calculate confidence based on proximity
        confidence = 1.0
        if question_bbox and answer_bbox:
            # Calculate vertical distance
            q_center_y = (question_bbox[1] + question_bbox[3]) / 2
            a_center_y = (answer_bbox[1] + answer_bbox[3]) / 2
            distance = abs(a_center_y - q_center_y)

            # Normalize confidence (closer = higher confidence)
            max_distance = 100  # pixels
            confidence = max(0.5, 1.0 - (distance / max_distance))

        return {
            'question': question,
            'answer': answer,
            'confidence': round(confidence, 2)
        }



    def build_structured_output(self, doc, layout_data, entities, key_value_pairs):
        """Build final structured output from all extracted information"""
        # Build structured data
        structured_data = {
            'document_id': doc['id'],
            #'document_type': doc_type,
            'entities': entities,
            'key_value_pairs': key_value_pairs,
            #'common_fields': common_fields,
            'layout_summary': {
                'total_words': len(doc['words']),
                'sections': {k: len(v['words']) for k, v in layout_data.items()}
            }
        }

        # Add metadata
        structured_data['metadata'] = {
            'extraction_time': datetime.now().isoformat(),
            'confidence': self.calculate_overall_confidence(entities, key_value_pairs),
            'source': 'FUNSD dataset',
            'processing_version': '1.0'
        }

        return structured_data


    def calculate_overall_confidence(self, entities, key_value_pairs):
        """Calculate overall confidence score"""

        # Base confidence
        confidence = 0.5

        # Boost for having entities
        if any(len(v) > 0 for v in entities.values()):
            confidence += 0.2

        # Boost for key-value pairs
        if key_value_pairs:
            confidence += 0.2

            # Additional boost for high-confidence pairs
            avg_pair_confidence = np.mean([p['confidence'] for p in key_value_pairs])
            confidence += avg_pair_confidence * 0.1

        return min(0.99, round(confidence, 2))

    def evaluate_extraction(self, doc_index=0, split='train'):
        """Evaluate extraction on a specific document"""

        doc = self.dataset[split][doc_index]

        print(f"\n{'='*70}")
        print(f"EVALUATING EXTRACTION ON DOCUMENT: {doc['id']}")
        print(f"{'='*70}")

        # Visualize the document
        self.visualize_document(doc)

        # Process the document
        result = self.process_document(doc)

        # Display results
        self.display_results(result)

        return result

    def display_results(self, result):
        """Display extraction results in a readable format"""

        print(f"\n DOCUMENT ANALYSIS RESULTS")
        print(f"{'='*50}")

        print(f"\n Document ID: {result['document_id']}")

        print(f"\n ENTITIES EXTRACTED:")
        print(f"{'-'*30}")
        for entity_type, entities_list in result['entities'].items():
            if entities_list:
                print(f"\n{entity_type.upper()}:")
                for i, entity in enumerate(entities_list[:5], 1):  # Show first 5
                    print(f"  {i}. {entity}")
                if len(entities_list) > 5:
                    print(f"  ... and {len(entities_list) - 5} more")

        print(f"\n KEY-VALUE PAIRS:")
        print(f"{'-'*30}")
        if result['key_value_pairs']:
            for i, pair in enumerate(result['key_value_pairs'][:10], 1):  # Show first 10
                print(f"\n  Pair {i}:")
                print(f"    Q: {pair['question'][:50]}...")
                print(f"    A: {pair['answer'][:50]}...")
                print(f"    Confidence: {pair['confidence']:.0%}")
        else:
            print("  No key-value pairs found")


        print(f"\n LAYOUT SUMMARY:")
        print(f"{'-'*30}")
        for section, count in result['layout_summary']['sections'].items():
            print(f"  {section}: {count} words")

        print(f"\n METADATA:")
        print(f"{'-'*30}")
        print(f"  Extraction Time: {result['metadata']['extraction_time']}")
        print(f"  Overall Confidence: {result['metadata']['confidence']:.0%}")

        # Create visualization
        self.create_result_visualization(result)

    def create_result_visualization(self, result):
          """Create a visualization of the extraction results"""

          fig, axes = plt.subplots(2, 2, figsize=(15, 12))
          axes = axes.flatten()

          # ===============================
          # 1. Entity Distribution
          # ===============================
          entity_counts = {k: len(v) for k, v in result["entities"].items()}

          if any(entity_counts.values()):
              labels = list(entity_counts.keys())
              counts = list(entity_counts.values())

              axes[0].bar(labels, counts)
              axes[0].set_title("Entity Distribution", fontsize=12)
              axes[0].set_ylabel("Count")
              axes[0].tick_params(axis="x", rotation=45)
              axes[0].set_xticklabels(labels, ha="right")
          else:
              axes[0].text(
                  0.5, 0.5, "No entities found",
                  ha="center", va="center",
                  transform=axes[0].transAxes
              )
              axes[0].axis("off")

          # ===============================
          # 2. Key-Value Pairs
          # ===============================
          if result["key_value_pairs"]:
              k = min(3, len(result["key_value_pairs"]))
              pair_text = "Top Key-Value Pairs:\n\n"

              for i, pair in enumerate(result["key_value_pairs"][:k], 1):
                  q = pair["question"][:30] + "..." if len(pair["question"]) > 30 else pair["question"]
                  a = pair["answer"][:30] + "..." if len(pair["answer"]) > 30 else pair["answer"]

                  pair_text += (
                      f"{i}. Q: {q}\n"
                      f"   A: {a}\n"
                      f"   Confidence: {pair['confidence']:.0%}\n\n"
                  )

              axes[1].text(
                  0.02, 0.98, pair_text,
                  fontsize=9,
                  va="top",
                  ha="left",
                  transform=axes[1].transAxes
              )
              axes[1].set_title("Key-Value Extraction", fontsize=12)
              axes[1].axis("off")
          else:
              axes[1].text(
                  0.5, 0.5, "No key-value pairs",
                  ha="center", va="center",
                  transform=axes[1].transAxes
              )
              axes[1].axis("off")

          # ===============================
          # 3. Layout Summary
          # ===============================
          layout_text = "Layout Analysis:\n\n"
          for section, count in result["layout_summary"]["sections"].items():
              layout_text += f"{section}: {count} words\n"

          axes[2].text(
              0.02, 0.98, layout_text,
              fontsize=10,
              va="top",
              ha="left",
              family="monospace",
              transform=axes[2].transAxes
          )
          axes[2].set_title("Document Layout", fontsize=12)
          axes[2].axis("off")

          # ===============================
          # 4. Disable unused subplot
          # ===============================
          axes[3].axis("off")

          # ===============================
          # Global layout fixes
          # ===============================
          plt.suptitle(
              f"Document Understanding Results: {result['document_id']}",
              fontsize=16,
              fontweight="bold"
          )

          plt.tight_layout(rect=[0, 0, 1, 0.95])
          plt.savefig(
              f"funsd_results_{result['document_id']}.png",
              dpi=150,
              bbox_inches="tight"
          )
          plt.show()

          print(f"Visualization saved as: funsd_results_{result['document_id']}.png")



In [10]:
def main():
    """Main execution function"""

    print("\n" + "="*70)
    print("FUNSD DOCUMENT UNDERSTANDING DEMONSTRATION")
    print("="*70)

    # Initialize processor
    processor = FUNSDDocumentProcessor()

    # Process multiple documents
    print("\n" + "="*70)
    print("PROCESSING FUNSD DOCUMENTS")
    print("="*70)

    results = []

    # Process 3 documents from training set
    for i in range(min(3, len(processor.dataset['train']))):
        print(f"\nProcessing document {i+1}/3...")

        # Get document
        doc = processor.dataset['train'][i]

        # Process document
        result = processor.evaluate_extraction(i, 'train')
        results.append(result)

    # Summary
    print("\n" + "="*70)
    print("SUMMARY OF RESULTS")
    print("="*70)

    total_entities = 0
    total_pairs = 0


    print(f"\n Extraction Statistics:")
    print(f"  Total entities extracted: {total_entities}")
    print(f"  Total key-value pairs: {total_pairs}")
    print(f"  Average entities per document: {total_entities/len(results):.1f}")
    print(f"  Average pairs per document: {total_pairs/len(results):.1f}")

    return processor, results

def batch_process(processor, num_docs=5):
    """Batch process multiple documents"""

    print(f"\n{'='*70}")
    print(f"BATCH PROCESSING {num_docs} DOCUMENTS")
    print(f"{'='*70}")

    batch_results = []

    for i in range(min(num_docs, len(processor.dataset['test']))):
        print(f"\nProcessing test document {i+1}/{num_docs}...")

        doc = processor.dataset['test'][i]

        # Quick processing without visualization
        result = processor.process_document(doc)
        batch_results.append(result)

        print(f"  Document: {doc['id']}")
        print(f"  Entities: {sum(len(v) for v in result['entities'].values())}")
        print(f"  Key-value pairs: {len(result['key_value_pairs'])}")

    return batch_results



In [ ]:
# Run main pipeline
processor, results = main()
# Optional: Batch process test documents
print("\n" + "="*70)
print("BATCH PROCESSING TEST SET")
print("="*70)

test_results = batch_process(processor, num_docs=3)

print("\n" + "="*70)
print("FUNSD PIPELINE COMPLETE!")
print("="*70)
print("\n Successfully processed FUNSD documents")
print(" Extracted structured information from unstructured forms")
print("Generated visualizations and summaries")
print("\n Outputs saved as PNG files in current directory")

## Training part

In [12]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.metrics import classification_report
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    DataCollatorForTokenClassification
)
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report

In [19]:
# Encoding function - FIXED VERSION
def encode_example(example):
    image = example["image"].convert("RGB")

    # Process with LayoutLMv3 processor
    encoding = processor(
        images=image,
        text=example["words"],
        boxes=example["bboxes"],
        padding="max_length",
        truncation=True,
        stride=128,  # Use the stride parameter
        return_overflowing_tokens=False,
        return_tensors="pt"
    )

    # Get word IDs
    word_ids = encoding.word_ids(batch_index=0)

    # Align labels with tokens
    labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != previous_word_id:
            # Get the label for the original word
            labels.append(example["ner_tags"][word_id])
        else:
            # Same word, subsequent subword
            labels.append(example["ner_tags"][word_id] if label_all_tokens else -100)
        previous_word_id = word_id

    # Remove batch dimension
    encoding = {k: v.squeeze(0) for k, v in encoding.items()}
    encoding["labels"] = torch.tensor(labels)

    return encoding

In [ ]:
# Label configuration
label_list = [
    "O",
    "B-QUESTION", "I-QUESTION",
    "B-ANSWER", "I-ANSWER",
    "B-HEADER", "I-HEADER"
]

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

# Initialize processor and model
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Load dataset
dataset = load_dataset("nielsr/funsd")
train_dataset = dataset["train"]
test_dataset = dataset["test"]


# Set label_all_tokens
label_all_tokens = True

# Encode datasets
train_dataset = train_dataset.map(
    encode_example,
    remove_columns=train_dataset.column_names
)

test_dataset = test_dataset.map(
    encode_example,
    remove_columns=test_dataset.column_names
)

# Set format to torch
columns = ["input_ids", "attention_mask", "bbox", "labels", "pixel_values"]
train_dataset.set_format(type="torch", columns=columns)
test_dataset.set_format(type="torch", columns=columns)


# DataLoader - WITHOUT collator since we already padded
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    # No collate_fn needed since we padded to max_length
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

In [ ]:


# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
epochs = 10
model.train()

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        # Move to device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            pixel_values=batch["pixel_values"],
            labels=batch["labels"]  # Let the model compute loss
        )

        loss = outputs.loss

        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

# Evaluation function
def evaluate_model(model, dataloader, device):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                bbox=batch["bbox"],
                pixel_values=batch["pixel_values"]
            )

            predictions = torch.argmax(outputs.logits, dim=-1)

            # Filter out ignored labels (-100): because it mean just "Ignore this token (special tokens, padding tokens, or subsequent subwords if label_all_tokens=False)""
            for i in range(predictions.shape[0]):
                mask = batch["labels"][i] != -100
                valid_predictions = predictions[i][mask].cpu().numpy()
                valid_labels = batch["labels"][i][mask].cpu().numpy()

                all_predictions.extend(valid_predictions)
                all_labels.extend(valid_labels)

    # Print classification report
    print("\n" + "="*50)
    print("Classification Report:")
    print("="*50)
    print(classification_report(
        all_labels,
        all_predictions,
        target_names=label_list,
        digits=4
    ))

    model.train()
    return all_predictions, all_labels

# Run evaluation
print("\nEvaluating on test set...")
predictions, labels = evaluate_model(model, test_loader, device)